# Finite State Projection (FSP)

Finite State Projection (FSP) for reaction networks.

Given a reaction network (a list of reactions, each with a stoichiometry
vector and a propensity function of the state) and a truncation order for
every species, build the projected CME generator A as a sparse matrix:
    dP / dt = A P,    P(t) in R^{|S|}, S = truncated state space.

Outgoing rates that target states outside S contribute to the diagonal of A
but not to any incoming entry; their integral over time is the "leak", a
diagnostic for truncation adequacy.

Two examples follow.

Example 1: single species, 0 -> X, 2X -> 0.
  Plot the evolution of P(X = n, t) as a 3D waterfall and compare leak vs t
  for several truncation orders.

Example 2: two species, gene expression
    0 -> M,     M -> 0,     M -> M + P,     P -> 0.
  Compute the joint distribution P(M, P, t = T) at a fixed time and show as
  a 2D heatmap with marginal distributions.

**Figures:** fsp_birth_death_leak.svg, fsp_birth_death_waterfall.svg, fsp_birth_death_waterfall_small.svg, fsp_gene_expression.svg, fsp_truncation_comparison.svg

In [ ]:
%matplotlib inline

In [ ]:
import itertools
import numpy as np
import scipy.sparse as sp
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.collections import PolyCollection


# ---------------------------------------------------------------------------
# General FSP framework
# ---------------------------------------------------------------------------
class Reaction:
    """A reaction with stoichiometry vector and propensity callable."""
    def __init__(self, stoich, propensity_fn, name=''):
        self.stoich        = np.array(stoich, dtype=int)
        self.propensity_fn = propensity_fn
        self.name          = name


def build_fsp_matrix(reactions, max_counts):
    """
    Build the FSP generator A and the enumerated state array.

    Arguments
    ---------
    reactions  : list[Reaction]
    max_counts : list[int]
        Per-species truncation; species i in {0, ..., max_counts[i]}.

    Returns
    -------
    A : csc_matrix of shape (n_states, n_states)
    state_array : ndarray of shape (n_states, n_species)
    """
    grids       = [np.arange(m + 1) for m in max_counts]
    state_array = np.array(list(itertools.product(*grids)))
    n_states    = len(state_array)
    state_to_ix = {tuple(s): i for i, s in enumerate(state_array)}
    max_arr     = np.array(max_counts)

    rows, cols, data = [], [], []
    for j, state in enumerate(state_array):
        total_out = 0.0
        for rxn in reactions:
            prop = rxn.propensity_fn(state)
            if prop <= 0.0:
                continue
            target = state + rxn.stoich
            if np.all(target >= 0) and np.all(target <= max_arr):
                rows.append(state_to_ix[tuple(target)])
                cols.append(j)
                data.append(prop)
            # Even if target is outside S, the rate still leaves state j;
            # it just isn't credited to any state inside S.
            total_out += prop
        if total_out > 0.0:
            rows.append(j); cols.append(j); data.append(-total_out)

    A = sp.csc_matrix((data, (rows, cols)), shape=(n_states, n_states))
    return A, state_array


def simulate_fsp(A, P0, t_eval):
    """Integrate dP/dt = A P, returning P with shape (n_states, n_times)."""
    def rhs(t, P):
        return A @ P
    sol = solve_ivp(rhs, [t_eval[0], t_eval[-1]], P0,
                    t_eval=t_eval, method='LSODA',
                    atol=1e-13, rtol=1e-10)
    return sol.y


# ---------------------------------------------------------------------------
# Plot style
# ---------------------------------------------------------------------------
plt.rcParams.update({
    'svg.fonttype':    'path',
    'mathtext.fontset':'cm',
    'font.family':     'serif',
    'font.size':        14,
    'axes.labelsize':   16,
    'axes.titlesize':   17,
    'legend.fontsize':  12,
    'xtick.labelsize':  13,
    'ytick.labelsize':  13,
    'lines.linewidth':  2.0,
})


# ===========================================================================
# Example 1: birth + dimer destruction
#   R1 :  0  -> X       rate k0
#   R2 :  2X -> 0       propensity k1 * n(n-1)/2
# Steady state ~ sqrt(k0/k1) molecules. With k0 = 5, k1 = 0.5, mean ~ 3.2.
# ===========================================================================
def example_1():
    print('Example 1: 0 -> X, 2X -> 0')

    k0 = 5.0
    k1 = 0.5

    # Reactions
    rxns = [
        Reaction([+1],
                 lambda s: k0,
                 '0 -> X'),
        Reaction([-2],
                 lambda s: k1 * s[0] * (s[0] - 1) / 2 if s[0] >= 2 else 0.0,
                 '2X -> 0'),
    ]

    # ----- Run at adequate truncation for the 3D waterfall -----
    N_main = 25
    A, states = build_fsp_matrix(rxns, [N_main])
    print(f'  waterfall: N_max = {N_main}, n_states = {len(states)}, '
          f'nnz(A) = {A.nnz}')

    P0 = np.zeros(len(states)); P0[0] = 1.0
    t_eval = np.linspace(0, 6, 13)         # 13 time slices for the waterfall
    P_sol  = simulate_fsp(A, P0, t_eval)
    print(f'  leak at adequate truncation: {1.0 - P_sol[:, -1].sum():.2e}')

    # ----- Run at several truncations to demonstrate the leak diagnostic -----
    truncations = [4, 5, 6, 8, 12]
    leak_t_eval = np.linspace(0, 6, 60)    # finer time grid for the leak panel
    leaks = {}
    for N in truncations:
        A_N, _ = build_fsp_matrix(rxns, [N])
        P0_N = np.zeros(N + 1); P0_N[0] = 1.0
        P_N = simulate_fsp(A_N, P0_N, leak_t_eval)
        leaks[N] = 1.0 - P_N.sum(axis=0)
        print(f'  N_max = {N:2d}: final leak = {leaks[N][-1]:.3e}')

    # ----- Figure 1: 3D waterfall (PolyCollection) -----
    fig    = plt.figure(figsize=(11, 8))
    ax_top = fig.add_subplot(111, projection='3d')
    cmap   = plt.get_cmap('viridis')
    colors = [cmap(i / max(len(t_eval) - 1, 1)) for i in range(len(t_eval))]

    n_grid = np.arange(N_main + 1)
    verts  = []
    for i in range(len(t_eval)):
        P_t = P_sol[:, i]
        x   = np.concatenate([[n_grid[0]], n_grid, [n_grid[-1]]])
        z   = np.concatenate([[0.0],         P_t,        [0.0]])
        verts.append(list(zip(x, z)))

    poly = PolyCollection(verts, alpha=0.85,
                          edgecolors='black', linewidth=0.4)
    poly.set_facecolor(colors)
    ax_top.add_collection3d(poly, zs=t_eval, zdir='y')

    ax_top.set_xlim(0, N_main)
    ax_top.set_ylim(t_eval[0], t_eval[-1])
    ax_top.set_zlim(0, P_sol.max() * 1.05)
    ax_top.set_xlabel(r'$n$  ($X$ count)', labelpad=8)
    ax_top.set_ylabel(r'Time  $t$',         labelpad=8)
    ax_top.set_zlabel(r'$P(X = n,\, t)$',   labelpad=8)
    ax_top.set_title(
        r'FSP solution:  $\varnothing \to X,\;\; 2X \to \varnothing$,  '
        rf'$N_{{\max}} = {N_main}$  (adequate)'
    )
    ax_top.view_init(elev=32, azim=-58)

    out_path = '/mnt/user-data/outputs/fsp_birth_death_waterfall.svg'
    plt.savefig(out_path, format='svg', bbox_inches='tight')
    plt.close()
    print(f'  saved {out_path}')

    # ----- Figure 2: probability leak vs time, several truncations -----
    fig, ax_bot = plt.subplots(figsize=(10, 5.5))
    leak_colors = ['#0072BD', '#D95319', '#EDB120', '#7E2F8E', '#77AC30']
    for (N, leak), color in zip(leaks.items(), leak_colors):
        ax_bot.semilogy(leak_t_eval, np.maximum(leak, 1e-16),
                        color=color, linewidth=2.0,
                        label=rf'$N_{{\max}} = {N}$')
    ax_bot.set_xlabel(r'Time  $t$')
    ax_bot.set_ylabel(r'Leak  $1 - \sum_n P_n$')
    ax_bot.set_title('Probability leak vs time, for several truncation orders')
    ax_bot.set_ylim(1e-13, 2.0)
    ax_bot.set_xlim(leak_t_eval[0], leak_t_eval[-1])
    ax_bot.grid(True, which='both', alpha=0.3)
    ax_bot.legend(loc='lower right', ncol=5, framealpha=0.95)
    for s in ('top', 'right'):
        ax_bot.spines[s].set_visible(False)

    out_path = '/mnt/user-data/outputs/fsp_birth_death_leak.svg'
    plt.savefig(out_path, format='svg', bbox_inches='tight')
    plt.close()
    print(f'  saved {out_path}')

    # ----- Figure 3: 3D waterfall at INADEQUATE truncation (N_max = 5) -----
    N_small = 5
    A_s, states_s = build_fsp_matrix(rxns, [N_small])
    P0_s = np.zeros(len(states_s)); P0_s[0] = 1.0
    P_sol_s = simulate_fsp(A_s, P0_s, t_eval)
    leak_s  = 1.0 - P_sol_s.sum(axis=0)
    print(f'  waterfall (inadequate): N_max = {N_small}, '
          f'final leak = {leak_s[-1]:.3e}')

    fig = plt.figure(figsize=(11, 8))
    ax3 = fig.add_subplot(111, projection='3d')
    n_grid_s = np.arange(N_small + 1)
    verts_s = []
    for i in range(len(t_eval)):
        P_t = P_sol_s[:, i]
        x   = np.concatenate([[n_grid_s[0]], n_grid_s, [n_grid_s[-1]]])
        z   = np.concatenate([[0.0], P_t, [0.0]])
        verts_s.append(list(zip(x, z)))

    poly_s = PolyCollection(verts_s, alpha=0.85,
                            edgecolors='black', linewidth=0.4)
    poly_s.set_facecolor(colors)
    ax3.add_collection3d(poly_s, zs=t_eval, zdir='y')

    ax3.set_xlim(0, N_small)
    ax3.set_ylim(t_eval[0], t_eval[-1])
    ax3.set_zlim(0, P_sol_s.max() * 1.05)
    ax3.set_xlabel(r'$n$  ($X$ count)', labelpad=8)
    ax3.set_ylabel(r'Time  $t$',         labelpad=8)
    ax3.set_zlabel(r'$P(X = n,\, t)$',   labelpad=8)
    ax3.set_title(
        r'FSP solution:  $\varnothing \to X,\;\; 2X \to \varnothing$,  '
        rf'$N_{{\max}} = {N_small}$  (inadequate, large leak)'
    )
    ax3.view_init(elev=32, azim=-58)

    out_path = '/mnt/user-data/outputs/fsp_birth_death_waterfall_small.svg'
    plt.savefig(out_path, format='svg', bbox_inches='tight')
    plt.close()
    print(f'  saved {out_path}')

    # ----- Figure 4: snapshot comparison N_max = 25 vs N_max = 5 -----
    snap_times = [0.5, 1.5, 3.0, 6.0]
    # Recompute both FSP solutions on a grid that includes the snapshot times
    t_snaps = np.sort(np.unique(np.concatenate([np.array(snap_times),
                                                 np.linspace(0, 6, 60)])))
    P_ref  = simulate_fsp(A,   np.concatenate([np.array([1.0]),
                               np.zeros(len(states) - 1)]), t_snaps)
    P_trunc = simulate_fsp(A_s, np.concatenate([np.array([1.0]),
                               np.zeros(len(states_s) - 1)]), t_snaps)

    fig, axes = plt.subplots(1, 4, figsize=(14, 4.2), sharey=True)
    fig.subplots_adjust(left=0.06, right=0.98, top=0.82, bottom=0.14,
                        wspace=0.10)
    fig.suptitle(
        r'FSP distribution snapshots:  adequate ($N_{\max}\!=\!25$)'
        r'  vs.  truncated ($N_{\max}\!=\!5$)',
        fontsize=14, fontweight='bold')

    BLUE = '#0072BD'; RED = '#D95319'
    for ax, t_snap in zip(axes, snap_times):
        i_snap = np.argmin(np.abs(t_snaps - t_snap))
        # Reference distribution (N_max = 25)
        P_r = P_ref[:, i_snap]
        ax.bar(np.arange(N_main + 1), P_r, width=0.9, align='center',
               color=BLUE, alpha=0.55, edgecolor='none',
               label=rf'$N_{{\max}} = {N_main}$')
        # Truncated distribution (N_max = 5)
        P_t = P_trunc[:, i_snap]
        ax.bar(np.arange(N_small + 1), P_t, width=0.55, align='center',
               color=RED, alpha=0.75, edgecolor='none',
               label=rf'$N_{{\max}} = {N_small}$')
        # Leak annotation
        leak_val = 1.0 - P_t.sum()
        ax.text(0.97, 0.95,
                f'leak $= {leak_val*100:.1f}$%',
                transform=ax.transAxes, fontsize=10,
                ha='right', va='top', color=RED,
                bbox=dict(fc='white', ec=RED, alpha=0.85, pad=3,
                          boxstyle='round,pad=0.3'))
        ax.set_xlabel(r'$n$')
        ax.set_title(rf'$t = {t_snap:g}$', fontsize=13)
        ax.set_xlim(-0.8, 14)
        for s in ('top', 'right'):
            ax.spines[s].set_visible(False)

    axes[0].set_ylabel(r'$P(X = n,\, t)$')
    axes[0].legend(loc='upper left', fontsize=10, framealpha=0.95)

    out_path = '/mnt/user-data/outputs/fsp_truncation_comparison.svg'
    plt.savefig(out_path, format='svg', bbox_inches='tight')
    plt.close()
    print(f'  saved {out_path}')


# ===========================================================================
# Example 2: gene expression
#   R1 :  0  -> M           rate alpha_m
#   R2 :  M  -> 0           rate beta_m  * n_M
#   R3 :  M  -> M + P       rate alpha_p * n_M
#   R4 :  P  -> 0           rate beta_p  * n_P
# ===========================================================================
def example_2():
    print('Example 2: gene expression')

    alpha_m = 1.0
    beta_m  = 1.0
    alpha_p = 5.0
    beta_p  = 0.1

    rxns = [
        Reaction([+1,  0], lambda s: alpha_m,         '0 -> M'),
        Reaction([-1,  0], lambda s: beta_m  * s[0],  'M -> 0'),
        Reaction([ 0, +1], lambda s: alpha_p * s[0],  'M -> M + P'),
        Reaction([ 0, -1], lambda s: beta_p  * s[1],  'P -> 0'),
    ]

    N_max_M = 12
    N_max_P = 150           # >= mean + ~6 sigma to make leak negligible

    A, states = build_fsp_matrix(rxns, [N_max_M, N_max_P])
    print(f'  N_max_M = {N_max_M}, N_max_P = {N_max_P}, '
          f'n_states = {len(states)}, nnz(A) = {A.nnz}')

    P0      = np.zeros(len(states)); P0[0] = 1.0
    T_final = 100.0
    P_sol   = simulate_fsp(A, P0, np.array([0.0, T_final]))
    P_final = P_sol[:, -1]
    leak    = 1.0 - P_final.sum()
    print(f'  leak at T = {T_final}: {leak:.2e}')

    joint = P_final.reshape(N_max_M + 1, N_max_P + 1)
    P_M   = joint.sum(axis=1)
    P_P   = joint.sum(axis=0)

    mean_M = np.sum(np.arange(N_max_M + 1) * P_M)
    mean_P = np.sum(np.arange(N_max_P + 1) * P_P)
    var_P  = np.sum((np.arange(N_max_P + 1) - mean_P)**2 * P_P)
    print(f'  <M> = {mean_M:.2f},  <P> = {mean_P:.2f},  '
          f'Fano(P) = {var_P/mean_P:.2f}')

    # ----- Plot: 2D heatmap with marginals using make_axes_locatable -----
    from mpl_toolkits.axes_grid1 import make_axes_locatable

    fig, ax_main = plt.subplots(figsize=(11, 8))

    im = ax_main.imshow(
        joint, origin='lower', cmap='viridis', aspect='auto',
        extent=[-0.5, N_max_P + 0.5, -0.5, N_max_M + 0.5],
        interpolation='nearest',
    )
    ax_main.set_xlabel(r'Protein count  $p$')
    ax_main.set_ylabel(r'mRNA count  $m$')
    ax_main.set_xlim(-0.5, N_max_P + 0.5)
    ax_main.set_ylim(-0.5, N_max_M + 0.5)

    divider  = make_axes_locatable(ax_main)
    ax_top   = divider.append_axes('top',    size='22%', pad=0.08,
                                   sharex=ax_main)
    ax_right = divider.append_axes('right',  size='22%', pad=0.08,
                                   sharey=ax_main)
    cax      = divider.append_axes('bottom', size='4%',  pad=0.55)

    # Top marginal P(p): bars centred on integer p values
    ax_top.bar(np.arange(N_max_P + 1), P_P, width=0.95, align='center',
               color='#0072BD', edgecolor='none')
    ax_top.set_ylabel(r'$P(p)$')
    ax_top.tick_params(axis='x', labelbottom=False)
    for s in ('top', 'right'):
        ax_top.spines[s].set_visible(False)

    # Right marginal P(m): bars centred on integer m values
    ax_right.barh(np.arange(N_max_M + 1), P_M, height=0.95, align='center',
                  color='#D95319', edgecolor='none')
    ax_right.set_xlabel(r'$P(m)$')
    ax_right.tick_params(axis='y', labelleft=False)
    for s in ('top', 'right'):
        ax_right.spines[s].set_visible(False)

    # Re-pin the limits AFTER all bar autoscaling has happened
    ax_main.set_xlim(-0.5, N_max_P + 0.5)
    ax_main.set_ylim(-0.5, N_max_M + 0.5)

    cbar = fig.colorbar(im, cax=cax, orientation='horizontal')
    cbar.set_label(r'$P(m, p)$')

    fig.suptitle(
        rf'Gene expression FSP:  $P(m, p)$ at $t = {T_final:g}$,  '
        rf'$\alpha_m={alpha_m:g}$, $\beta_m={beta_m:g}$, '
        rf'$\alpha_p={alpha_p:g}$, $\beta_p={beta_p:g}$',
        fontsize=14,
    )

    out_path = '/mnt/user-data/outputs/fsp_gene_expression.svg'
    plt.savefig(out_path, format='svg', bbox_inches='tight')
    plt.close()
    print(f'  saved {out_path}')


if __name__ == '__main__':
    example_1()
    print()
    example_2()